# 多层感知机与学习规则

本节先在 MNIST 上训练单层分类器，再加入隐藏层，得到多层感知机（MLP）。接着手工计算一次反向传播（BP）中的梯度，比较激活函数，并尝试用 Hebb 规则学习图像特征。

这几组实验分别关注网络结构、激活函数和权重更新方式。比较结果时，可以先看各组改变了什么，再看学习曲线和分类准确率。

[本课说明与运行步骤](README.md)


In [ ]:
from biai.paths import DATA_DIR
from biai.reproducibility import seed_everything

SEED = 0
seed_everything(SEED, deterministic=True)

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import random
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import Subset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

## 准备 MNIST

每张图像是 28 × 28 的灰度图，标签为数字 0–9。`ToTensor()` 将像素缩放到 [0, 1]，模型再把图像展平为 784 维向量。从原训练集固定留出 10% 作为验证集，用于观察每轮学习效果；其余样本按批次打乱训练。测试集只在各模型训练结束后评估一次。数据首次使用时下载到仓库的 `data/`。

In [ ]:
batch_size = 64
transform = transforms.Compose([
     transforms.ToTensor()])

train_dataset = datasets.MNIST(root=DATA_DIR, train=True, download=True, transform=transform)
indices = torch.randperm(len(train_dataset), generator=torch.Generator().manual_seed(SEED))
validation_size = len(train_dataset) // 10
validation_indices = indices[:validation_size].tolist()
training_indices = indices[validation_size:].tolist()
val_dataset = Subset(train_dataset, validation_indices)
train_dataset = Subset(train_dataset, training_indices)
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True,
    generator=torch.Generator().manual_seed(SEED)
)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

test_dataset = datasets.MNIST(root=DATA_DIR, train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
print(f"数据集大小: {len(train_dataset)}")
print(f"图像形状: {train_dataset[0][0].shape}")

In [ ]:
idx = random.sample(range(len(train_dataset)), 1)
image, target = train_dataset[idx[0]]
print(f"target: {target}")

# Convert from CHW to HWC for plotting.
image_np = image.numpy().transpose(1, 2, 0)

fig, ax1 = plt.subplots(1, 1, figsize=(12, 5))
ax1.imshow(image_np.squeeze(), cmap='gray')


## 用反向传播训练

网络为每张图像输出 10 个类别分数（logits），交叉熵损失根据这些分数和真实标签计算误差。一次参数更新依次执行：清空上一批次的梯度、前向计算、反向求导、优化器更新。每轮训练结束后，记录训练集与验证集的交叉熵和准确率。训练指标汇总该轮更新过程中的所有批次，验证指标使用轮末参数。

### 单层线性分类器

输入直接经过一个线性层得到 10 个分数。这个模型没有隐藏层，可作为后续比较的起点。`CrossEntropyLoss` 直接接收分数，模型末尾无需添加 softmax。


In [ ]:
class MLP_one_linear(nn.Module):
    def __init__(self, input_dim=784, output_dim=10):
        super(MLP_one_linear, self).__init__()
        # Map 784 pixels to 10 class scores.
        self.fc1 = nn.Linear(input_dim, output_dim)

    def forward(self, x):
        # Flatten each image while keeping the batch dimension.
        x = x.view(x.size(0), -1)
        y = self.fc1(x)
        return y

seed_everything(SEED, deterministic=True)
model = MLP_one_linear(input_dim=784, output_dim=10).to(device)
criterion = nn.CrossEntropyLoss()   # Takes logits, so no separate softmax is needed.
optimizer = optim.SGD(model.parameters(), lr=0.05)

In [ ]:
def evaluate(model, loader):
    model.eval()
    loss_sum, correct, count = 0.0, 0, 0
    with torch.no_grad():
        for x, target in loader:
            x, target = x.to(device), target.to(device)
            logits = model(x)
            loss_sum += nn.functional.cross_entropy(logits, target, reduction="sum").item()
            correct += (logits.argmax(1) == target).sum().item()
            count += target.numel()
    return loss_sum / count, correct / count


def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    for split in ("train", "validation"):
        axes[0].plot([row[f"{split}_loss"] for row in history], label=split)
        axes[1].plot([row[f"{split}_accuracy"] for row in history], label=split)
    axes[0].set_ylabel("Cross entropy")
    axes[1].set_ylabel("Accuracy")
    for ax in axes:
        ax.set_xlabel("Epoch (from 0)")
        ax.legend()
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def train_classifier(model, optimizer, epochs):
    train_loader.generator.manual_seed(SEED)
    history = []
    for epoch in range(epochs):
        model.train()
        loss_sum, correct, count = 0.0, 0, 0
        for x, target in train_loader:
            x, target = x.to(device), target.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, target)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * target.numel()
            correct += (logits.argmax(1) == target).sum().item()
            count += target.numel()
        val_loss, val_acc = evaluate(model, val_loader)
        history.append(dict(train_loss=loss_sum / count, train_accuracy=correct / count,
                            validation_loss=val_loss, validation_accuracy=val_acc))
        print(f"Epoch {epoch+1}: train loss={loss_sum/count:.4f}, acc={correct/count:.2%}; "
              f"validation loss={val_loss:.4f}, acc={val_acc:.2%}")
    test_loss, test_acc = evaluate(model, test_loader)
    print(f"Final test: loss={test_loss:.4f}, accuracy={test_acc:.2%}")
    plot_history(history, type(model).__name__)
    return history, test_acc


epochs = 10
linear_history, linear_test_accuracy = train_classifier(model, optimizer, epochs)

### 带一个隐藏层的 MLP

在输入和输出之间加入 128 个隐藏单元，并使用 ReLU 激活。若去掉 ReLU，两个线性层的组合仍然只能表示一个线性变换。此处沿用上一节的学习率、批次大小和 10 轮训练预算，比较两种模型的验证曲线及最终测试准确率。

In [ ]:
class MLP_two_linear(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=128, output_dim=10):
        super(MLP_two_linear, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)   # 784 inputs -> 128 hidden units.
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        h = torch.relu(self.fc1(x))      # ReLU makes the two-layer model nonlinear.
        y = self.fc2(h)
        return y

seed_everything(SEED, deterministic=True)
model = MLP_two_linear().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.05)
epochs = 10
mlp_history, mlp_test_accuracy = train_classifier(model, optimizer, epochs)

## 手工计算反向传播

前面的模型通过 `loss.backward()` 自动求导。下面把两层权重的更新展开：先计算输出误差，再通过输出层权重将误差传回隐藏层。两层梯度都使用更新前的权重，最后一起更新。

权重变化写成“误差信号 × 输入活动”的形式，但隐藏层仍需要来自输出层的误差信号，因此属于 BP。后面的 Hebb 示例会展示只依靠输入和隐藏活动的局部更新。

为简化求导，这个手工示例省略了偏置，也采用了不同的初始化方式。它适合用来理解梯度计算；学习规则的效果留到后面的相同初始化对照中比较。


In [ ]:
import torch
import torch.nn.functional as F
from torchvision import datasets, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

batch_size = 64
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))  # Flatten each image before batching.
])

train_dataset = datasets.MNIST(root=DATA_DIR, train=True, download=True, transform=transform)
val_dataset = Subset(train_dataset, validation_indices)
train_dataset = Subset(train_dataset, training_indices)
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True,
    generator=torch.Generator().manual_seed(SEED)
)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

test_dataset = datasets.MNIST(root=DATA_DIR, train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

input_dim = 784      # 28 x 28 pixels.
hidden_dim = 128     # Hidden units.
output_dim = 10      # Digit classes.
eta = 0.05           # Learning rate.

In [ ]:
# Scale each layer by its fan-in; this example has no biases.
seed_everything(SEED, deterministic=True)
W1 = torch.randn(hidden_dim, input_dim, device=device) * (2 / input_dim) ** 0.5
W2 = torch.randn(output_dim, hidden_dim, device=device) * (1 / hidden_dim) ** 0.5

def forward(x):
    h = torch.relu(torch.matmul(x, W1.T))
    y = torch.matmul(h, W2.T)
    return h, y

def manual_backprop_update(x, h, y, target):
    global W1, W2
    target_onehot = F.one_hot(target, num_classes=output_dim).float()

    y_pred = torch.softmax(y, dim=1)

    # Average the outer products of output errors and hidden activations.
    error_out = target_onehot - y_pred
    dW2 = eta * torch.matmul(error_out.T, h)
    # Use the same pre-update weights for both layer gradients.
    error_hidden = torch.matmul(error_out, W2) * (h > 0).float()
    dW1 = eta * torch.matmul(error_hidden.T, x)
    W1 += dW1 / x.size(0)
    W2 += dW2 / x.size(0)

In [ ]:
def evaluate_manual(loader):
    loss_sum, correct, count = 0.0, 0, 0
    for x, target in loader:
        x, target = x.to(device), target.to(device)
        _, logits = forward(x)
        loss_sum += F.cross_entropy(logits, target, reduction="sum").item()
        correct += (logits.argmax(1) == target).sum().item()
        count += target.numel()
    return loss_sum / count, correct / count


epochs = 10
manual_history = []
for epoch in range(epochs):
    loss_sum, correct, count = 0.0, 0, 0
    for x, target in train_loader:
        x, target = x.to(device), target.to(device)
        h, y = forward(x)
        loss_sum += F.cross_entropy(y, target, reduction="sum").item()
        correct += (y.argmax(1) == target).sum().item()
        count += target.numel()
        manual_backprop_update(x, h, y, target)
    val_loss, val_acc = evaluate_manual(val_loader)
    manual_history.append(dict(train_loss=loss_sum/count, train_accuracy=correct/count,
                               validation_loss=val_loss, validation_accuracy=val_acc))
    print(f"Epoch {epoch+1}: train loss={loss_sum/count:.4f}, acc={correct/count:.2%}; "
          f"validation loss={val_loss:.4f}, acc={val_acc:.2%}")
manual_test_loss, manual_test_accuracy = evaluate_manual(test_loader)
print(f"Final test: loss={manual_test_loss:.4f}, accuracy={manual_test_accuracy:.2%}")
plot_history(manual_history, "Manual gradient updates")

## 对照一：激活函数

ReLU 将负值置零，Sigmoid 把输出压到 0–1，Tanh 把输出压到 −1–1。下面只改变隐藏层的激活函数，比较三种情况下的学习速度和分类准确率。

三个网络均为 784→128→10，复制相同的初始权重与偏置，使用相同的数据划分、每轮样本顺序和训练轮数。优化器统一采用 SGD，学习率为 0.05。因此，这组结果反映的是三种激活函数在该学习率下的表现。


In [ ]:
class ActivationMLP(nn.Module):
    def __init__(self, activation="relu", bias=True):
        super().__init__()
        self.fc1 = nn.Linear(784, 128, bias=bias)
        self.fc2 = nn.Linear(128, 10, bias=bias)
        self.activation = {"relu": nn.ReLU(), "sigmoid": nn.Sigmoid(), "tanh": nn.Tanh()}[activation]

    def forward(self, x):
        return self.fc2(self.activation(self.fc1(x.flatten(1))))


seed_everything(SEED, deterministic=True)
activation_initial = ActivationMLP().state_dict()
activation_results = {}
for activation in ("relu", "sigmoid", "tanh"):
    seed_everything(SEED, deterministic=True)
    activation_model = ActivationMLP(activation).to(device)
    activation_model.load_state_dict(activation_initial)
    history, score = train_classifier(
        activation_model, optim.SGD(activation_model.parameters(), lr=0.05), epochs
    )
    activation_results[activation] = dict(history=history, test_accuracy=score)
for name, result in activation_results.items():
    print(f"{name:8s} validation={result['history'][-1]['validation_accuracy']:.2%}, "
          f"test={result['test_accuracy']:.2%}")


## 对照二：用 Hebb 规则学习特征

Hebb 规则根据输入活动 $x$ 与神经元响应 $y$ 的共同出现增强连接。直接累加 $yx$ 会使权重不断增大，[Oja 规则](https://users.ics.aalto.fi/oja/papers.html) 为此加入了与 $y^2w$ 成比例的稳定项。

如果多个隐藏单元学到了相似的方向，增加单元数量也未必带来更多有用特征。[Sanger 的广义 Hebb 规则（GHA）](https://proceedings.neurips.cc/paper_files/paper/1988/file/e00da03b685a0dd18fb6a08af0923de0-Paper.pdf) 在更新中扣除此前单元已表示的分量，鼓励不同单元学习不同方向。Oja 与 GHA 都只用输入和隐藏活动更新特征层，不需要标签或输出层传回的误差。

下面用四个结构相同、均不含偏置的网络作比较：

| 方法 | 特征层如何更新 | 分类层如何更新 |
| --- | --- | --- |
| BP | 根据分类误差反向传播 | 根据分类误差更新 |
| Oja | 根据输入和隐藏活动作局部更新 | 根据分类误差更新 |
| GHA | 在局部更新中减少特征重复 | 根据分类误差更新 |
| 固定随机特征 | 保持初始权重 | 根据分类误差更新 |

四个网络复制相同的初始参数，读取相同批次，分类层学习率均为 0.05；Oja 与 GHA 的特征层学习率为 0.001。局部更新使用 ReLU 之前的线性响应，分类层读取 ReLU 之后的活动。固定随机特征这一组可帮助判断：特征学习是否比直接使用随机特征更有用？

Oja 与 GHA 没有直接优化分类误差，所以要结合分类准确率判断它们学到的特征是否适合数字识别。表格中的隐藏权重平均绝对余弦用于观察特征重复程度：接近 1 表示方向重复，接近 0 表示方向接近正交。

本页输入没有逐像素减去训练集均值，GHA 对应的是未中心化数据的二阶矩方向。更新公式见下面的附录。


In [ ]:
@torch.no_grad()
def oja_update(layer, x, learning_rate):
    """Update a bias-free feature layer using activities, without labels or feedback."""
    x = x.flatten(1)
    y = x @ layer.weight.T
    correlation = y.T @ x / x.shape[0]
    stabilization = y.square().mean(0)[:, None] * layer.weight
    layer.weight.add_(learning_rate * (correlation - stabilization))


@torch.no_grad()
def generalized_hebbian_update(layer, x, learning_rate):
    """Use Sanger's lower-triangular activity term to learn distinct directions."""
    x = x.flatten(1)
    y = x @ layer.weight.T
    correlation = y.T @ x
    decorrelation = torch.tril(y.T @ y) @ layer.weight
    layer.weight.add_(learning_rate * (correlation - decorrelation) / x.shape[0])


def train_learning_rule(model, rule, epochs):
    feature_learning = rule == "BP"
    model.fc1.weight.requires_grad_(feature_learning)
    parameters = model.parameters() if feature_learning else model.fc2.parameters()
    optimizer = optim.SGD(parameters, lr=0.05)
    train_loader.generator.manual_seed(SEED)
    history = []
    for epoch in range(epochs):
        model.train()
        loss_sum, correct, count = 0.0, 0, 0
        for x, target in train_loader:
            x, target = x.to(device), target.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = F.cross_entropy(logits, target)
            loss.backward()
            optimizer.step()
            if rule == "Oja":
                oja_update(model.fc1, x, learning_rate=0.001)
            elif rule == "GHA":
                generalized_hebbian_update(model.fc1, x, learning_rate=0.001)
            loss_sum += loss.item() * target.numel()
            correct += (logits.argmax(1) == target).sum().item()
            count += target.numel()
        val_loss, val_acc = evaluate(model, val_loader)
        history.append(dict(train_loss=loss_sum/count, train_accuracy=correct/count,
                            validation_loss=val_loss, validation_accuracy=val_acc))
        print(f"{rule} epoch {epoch+1}: validation={val_acc:.2%}")
    _, score = evaluate(model, test_loader)
    plot_history(history, rule)
    directions = F.normalize(model.fc1.weight.detach(), dim=1)
    similarity = (directions @ directions.T).abs()
    off_diagonal = ~torch.eye(len(directions), dtype=torch.bool, device=device)
    return dict(history=history, test_accuracy=score,
                mean_abs_feature_cosine=similarity[off_diagonal].mean().item())


seed_everything(SEED, deterministic=True)
rule_initial = ActivationMLP(bias=False).state_dict()
rule_results = {}
for rule in ("BP", "Oja", "GHA", "Frozen"):
    seed_everything(SEED, deterministic=True)
    rule_model = ActivationMLP(bias=False).to(device)
    rule_model.load_state_dict(rule_initial)
    rule_results[rule] = train_learning_rule(rule_model, rule, epochs)
for name, result in rule_results.items():
    print(f"{name:8s} validation={result['history'][-1]['validation_accuracy']:.2%}, "
          f"test={result['test_accuracy']:.2%}, feature |cos|={result['mean_abs_feature_cosine']:.4f}")


## 可选扩展

- 可以让 GHA 的特征学习率逐轮衰减，观察特征方向的重复程度和分类准确率如何变化。
- 可以把 BP 的 SGD 换成带动量的 SGD，保持网络和训练轮数一致，比较学习曲线。超参数可在验证集上选择，测试集用于最后评估。

### 附录：Oja 与 GHA 的更新量

单个样本的更新为 $\Delta w_j=\eta(y_jx-y_j^2w_j)$，其中 $y_j=w_j^Tx$。代码对批次取平均。式中没有标签、输出误差或分类层权重，这正是它与前面手工 BP 更新的区别。

GHA 将 Oja 的稳定项扩展为 $\sum_{k\leq j}y_jy_kw_k$，因此 $\Delta w_j=\eta[y_jx-\sum_{k\leq j}y_jy_kw_k]$。`tril()` 保留下三角，对应每个单元扣除自身与此前单元已表示的方向。分类层依旧读取 ReLU 后的活动；特征更新使用线性响应 $y$。